<h1>Section 1: Password Strength Analysis</h1>

In [ ]:
import string
import math

# Prompt the user to enter a password for analysis
password = input("input password here ->").strip()
# Determine the character pool size based on which 
# characters appear
pool_size =0
# Add lowercase letters to the pool size
if any(c.islower() for c in password):
    pool_size+=26
# Add uppercase letters to the pool size
if any(c.isupper() for c in password):
    pool_size+=26
# Add digits to the pool size
if any(c.isdigit() for c in password):
    pool_size+=10
# Add punctuation/special characters to the pool size
if any(c in string.punctuation for c in password):
    pool_size += len(string.punctuation)
# calculate entropy = length × log2(pool size)
entropy= len(password) * math.log2(pool_size)

# password strength based on entropy score
if entropy < 28:
    print("Very weak")
elif entropy < 36:
    print("Weak")
elif entropy < 60:
    print("Reasonable")
elif entropy < 128:
    print("Strong")
else:
    print("Very strong")
# Display the entered password for reference
print('Your password is:  ' + password)
with open("cmn_pwds.txt", "r") as common_password:
    for word in common_password:
        if password == word.strip():
            # Check against a list of commonly used passwords
            print("password is common")
            break
    else:
        print('password is not common')
# Evaluate whether the password meets length requirements
if len(password) > 16:
    print("execellent length")
if len(password) > 12:
    print("Good length")
else:
    print("Weak length")
# checks for missing lower or uppercase and checks for special
# characters.
if password.islower():
    print("Please add upper case characters")
if password.isupper():
    print("Plese include lower case characters")

if any(c in string.punctuation for c in password):
    print("contains special characters")
else:
    print("does not contain special characters")



Very weak
Your password is:  fgarg
password is not common
Weak length
Please add upper case characters
does not contain special characters


<h1>Section 2: Dont store passowords, store hashes!</h1>

<h3>Hash with MD5</h3>

In [2]:
# imports hashlib which provides the hashing algorithms
import hashlib  
# creates a function that hashes a password using MD5 and
# returns the hash in a hexadecimal format
def hash_with_MD5(password:str) -> str:
    return hashlib.md5(password.encode()).hexdigest()
# call the function with password of choice 
password = hash_with_MD5("password123")
print(password)


482c811da5d5b4bc6d497ffa98491e38


<h3>Hash with SHA256</h3>

In [ ]:
# Same as above code but using a different hashing algorithm
import hashlib  

def hash_with_sha(password:str) -> str:
    return hashlib.sha256(password.encode()).hexdigest()
    
password = hash_with_sha("password123")
print(password)

ef92b778bafe771e89245b89ecbc08a44a4e166c06659911881f383d4473e94f


<h3>Hash with bcrypt</h3>

In [ ]:
# Uses bcrypt library
import bcrypt

# function to hash a passowrd
def hash_password(password: str):
    # generates a random salt value with a cost factor of 12
    salt = bcrypt.gensalt(rounds=12)
    print(f'salt -> {salt}')
    # displays the salt just for demonstration purposes
    password = bcrypt.hashpw(password.encode(),salt)
    # returns hashed password
    return password

# decided to print it twice in order to show that no two passwords
# provide the same output password value because salting is added.
print(hash_password('password123'))
print(hash_password('password123')) 


salt -> b'$2b$12$zQzNBzCGSxmYKGxBXNI8a.'
b'$2b$12$zQzNBzCGSxmYKGxBXNI8a.24tvFJ1PBmOBaoo8CxBX.OrnIZ4PObu'
salt -> b'$2b$12$OESxJGPoGNbL6Y6TCycA0u'
b'$2b$12$OESxJGPoGNbL6Y6TCycA0uh1fYtGQEJRCmbx7MtMeHkkesSg57m5W'


<h3>Hashing using salt and comparing two different hashes output to verify password</h3>

In [3]:
import bcrypt
# uses the bcrypt library

password = "password123" 
# hashes the password using the password and salt
stored_hash = bcrypt.hashpw(password.encode(), bcrypt.gensalt())

# a function that takes the regular password and hash and returns either true or false
def verify_password(password:str, stored_hash:bytes) -> bool:
    check_password = bcrypt.checkpw(password.encode(), stored_hash)
    return check_password

# the function is than called twice, one with the incorrect password 
# and one with the correct password to show that the function works as
# intended.
print(verify_password("password123", stored_hash))  
print(verify_password("wrongpass", stored_hash))  

True
False


<h1>Section 3: Adding Salt and Pepper</h1>

<h3>Hasing with salt</h3>

In [ ]:
import bcrypt 

# A function that is called twice with the same value to 
# show that no two pairs of a password is the same despite the passwords raw
# value being the exact time in this case 'user123password'.
def salty(password:str):
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt())
print(salty("user123password"))
print(salty("user123password"))


b'$2b$12$ZX/PqgCKju5oZPzuXCorLuqlniv7lFd1rWxOFWELmgulcmN68vcUW'
b'$2b$12$mbZI/1TbThl.qtRJspMagee52zYya6.VY6xXvlfoSqzYrPf.ytVSS'


<h3> Hashing without salt</h3>

In [1]:
import hashlib
# A function that does not use salt and it calls the same function twice to show that
# without salt, the two hash values would be the same since there is a lack 
# of salt.
def no_salt(password):      
    return hashlib.sha256(password.encode()).hexdigest()
    

password = "user123password"
hash1 = no_salt(password)
hash2 = no_salt(password)

print(" hash 1: ",hash1)
print(" hash 2: ",hash2)



 hash 1:  6384e02d9a47ac61c64950915c878ea54cc91a3d9c11d7497818017db8356d2a
 hash 2:  6384e02d9a47ac61c64950915c878ea54cc91a3d9c11d7497818017db8356d2a


<h3> Hashing with salt and pepper</h3>

In [ ]:
import bcrypt
import os
# This function utilises the salt and pepper factor. Pepper is a secret value
# which is attached two the password which is than hashed with the salt in
# order to provide optimal encryption.

# Define the pepper value from environment variable
pepper = os.environ.get("pepper", "mysecretpepperkey")

# Function to use salt and pepper
def hash_pw_pepper(password:str ) -> bytes:
    new= (password + pepper)
    print(new)
    # adds the pepper string value to the password and than encodes it altogether
    pepper_pw= (password + pepper).encode()
    # Final value is than hashed
    hashed = bcrypt.hashpw(pepper_pw, bcrypt.gensalt(rounds=12))
    return hashed

password = "user123password"

stored_hash = hash_pw_pepper(password)
print(stored_hash.decode())

user123passwordsecretpeppervalue
$2b$12$lefCUCPUFaFosGh4Et91fODifQdklaiQ9d3t9FqGaNb5LBOgw8Tzu


<h1>Section 4: Implementing Two-Factor Authentication</h1>

<h3>Server Side</h3>

In [ ]:
# Libaries for generating one time passwords, qr codes and time
import pyotp
import qrcode
import time
# creates a random 32 character base32 secret key for totp
code = pyotp.random_base32()
print("Code:  ", code)
# creates a TOTP object using the secret key
totp = pyotp.TOTP(code)
# Provides a URI that authenticator apps can than scan to set up the TOTP
uri = totp.provisioning_uri(name="user@example.com", issuer_name="MySecureApp")
print("URI:", uri)
# Creates a QR code from the URI and saves it as an image
qrcode.make(uri).save("totp.png")
print("QR code saved as totp.png")

Code:   QDMB3C3EI4C7JSCNLOXBR3P2VTP6Y3XQ
URI: otpauth://totp/MySecureApp:user%40example.com?secret=QDMB3C3EI4C7JSCNLOXBR3P2VTP6Y3XQ&issuer=MySecureApp
QR code saved as totp.png


In [ ]:
import pyotp
import time
import qrcode

# In the function above, the totp was already created 

# prints the current 6 digit totp code generates from the shared secret

print("Current TOTP:", totp.now())  
# asks the user to enter the code from their authenticator app
user_input = input("Enter the 6-digit code app: ")

# verifys if the entered code matches the current valid TOTP
if totp.verify(user_input):
    print("Code verified successfully!")
else:
    print(" Invalid code.")



Current TOTP: 612331
Code verified successfully!


<h1>(Challenge) Section 5: Simulating a Brute-Force Attack </h1>

In [ ]:
import hashlib
import time

password = "password"  # target password to crack
# creates a hashed password
target_hash = hashlib.sha256(password.encode()).hexdigest()

# opens the cmn_pwds text file and reads the file
with open('cmn_pwds.txt','r') as file:
    lines = file.read().split()
# A timer than starts to emasure how long the brute forcer takes 
start_time = time.time()
# a for loopthat goes down through each word and converts it to a hash
for line in lines:
    line = hashlib.sha256(line.encode()).hexdigest()
    # The hash is than printed for demonstration purposes to show the brute forcer
    # produce a hash.
    print(line)
    # if the target hash mataches the line than the applicaiton is succesful
    # the brute forcer and the timer both stop.
    if line == target_hash:
        end = time.time()
        print("Password found in wordlist!")
        final = end-start_time
        print(final)
        break



8d969eef6ecad3c29a3a629280e686cf0c3f5d5a86aff3ca12020c923adc6c92
ef797c8118f02dfb649607dd5d3f8c7623048c9c063d532cc95c5ed7a898a64f
65e84be33532fb784c48129675f9eff3a682b27168c0ea744b2cf58ee02337c5
15e2b0d3c33891ebb0f1ef609ec419420c20e320ce94c65fbc8c3312448eb225
5994471abb01112afcc18159f6cc74b4f511b99806da59b3caf5a9c173cacfc5
ab73d590e57e04c18fbe4c85a0983028927e8ba7b4cf74b0d076c52e979d4ef2
a5b810a3190a39033de4d82052fdf6f4c9765516d6b7eeb0c496adf8a3d3efc9
d2d02ea74de2c9fab1d802db969c18d409a8663a9697977bb1c98ccdd9de4372
c0c4a69b17a7955ac230bfc8db4a123eaa956ccf3c0022e68b8d4e2f5b699d1f
fa1baeb8e6f5c28f26997f63cc08bf08ff2632a58a578b8b971dd24d5c7d7863
db5e18fae9a82460d60d31eb30011d391c3d07a1c1f12bae0f2495b6b7e4f4ed
4c6eb87b502e3e019acbd4b1e579bd1566104abc0914f5186df63e4833c993c2
044f4b3501cd8e8131d40c057893f4fdff66bf4032ecae159e0c892a28cf6c8e
3fd30542fe3f61b14bd4a4b2dc0b6fb30fa6f63ebce52dd1778aaa8c4dc02cff
0e1d59321a7eb0e3bb9584245747cc73a9d03cd9ea2c24c39a071912dfadb33a
3bdcdc469bf1e4961d4b5053b

<h1>Section 6: Building the Complete System</h1>

In [ ]:
import string
import math
import bcrypt
import pyotp

# A dictionary to store the user account data in memory containing:
# - hashed password
# - totp secret for two-factor authentication
users = {}

def register_user():
    # The user inputs their data
    username = input("username -->")
    # If the username already exists than the user would have to input a new username
    if username in users:
        print("Username Exists")
        return False

    while True:
        # While loop so that until the user inputs a strong password
        # the program will keep trying
        password = input("password -->")
        strong = True
        pool_size =0

        if any(c.islower() for c in password):
            pool_size+=26
        if any(c.isupper() for c in password):
            pool_size+=26
        if any(c.isdigit() for c in password):
            pool_size+=10
        if any(c in string.punctuation for c in password):
            pool_size += len(string.punctuation)

        entropy= len(password) * math.log2(pool_size)

        if entropy < 36:
            print("Very weak")
            strong = False
        elif entropy < 36:
            print("Weak")
            strong = False
        elif entropy < 60:
            print("Reasonable")
        elif entropy < 128:
            print("Strong")
        else:
            print("Very strong")

        with open("cmn_pwds.txt", "r") as common_password:
            if any(password == word.strip() for word in common_password):
                    print("password is common")
                    strong = False
            else:
                print('password is not common')


        if len(password) > 16:
            print("execellent length")
        if len(password) > 12:
            print("Good length")
        else:
            print("Weak length")
            strong = False
            
        if password.islower():
            print("Please add upper case characters")
            strong = False
        if password.isupper():
            print("Plese include lower case characters")
            strong = False

        if any(c in string.punctuation for c in password):
            print("contains special characters")
        else:
            print("does not contain special characters")
            strong = False

        if strong:
            print(f"Password is accepted for {username}")
            break
        print("password too weak please try again")
    # Hashes the final password using bcrypty for secure storage
    hashed_password = bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()
    
    # Generates a TOTP for 2fa
    totp_secret = pyotp.random_base32()
    # Saves the user account data
    users[username] = {
        "password_hash":hashed_password,
        "totp_secret": totp_secret
    }
    # Shows the totp secret to the user so they can add it to the authenticator
    # app.
    print(f"\n  '{username}' registered successfully!")
    print(f" Save this secret for your TOTP app: {totp_secret}")
    return True 

def authenticate():
    # Asks for username
    username = input("Login Username:  ")
    # Check that the username exists in the system
    if username not in users:
        print("Username not found. Please try again.")
        return False
    # Asks for the password
    password = input("Password:  ")
    # Gets the store hash and compares it using bcrypt
    stored_hash = users[username]["password_hash"].encode()

    if not bcrypt.checkpw(password.encode(), stored_hash):
        print("Incorrect password entered.")
        return False
    
    print("Credentials verified successfully.")
    # performs 2FA 
    totp_secret = users[username]["totp_secret"]
    totp = pyotp.TOTP(totp_secret)
    otp = input("Enter 2FA code from your authenticator app: ")
    # Verifies the time based one time password
    if totp.verify(otp):
        print("Access granted. Welcome!")
        return True
    else:
        print("Invalid 2FA token. Access denied.")
        return False

if __name__ == "__main__":
    register_user()
    authenticate()



Strong
password is not common
execellent length
Good length
contains special characters
Password is accepted for user123

  'user123' registered successfully!
 Save this secret for your TOTP app: BQDC7S6TBTS2T4LFUC63SNR3NAOESRA3
Correct Credentials!
Access Granted  ٩(◕‿◕)۶ 


<h1>(Challenge) Section 7: Defence against brute forcer login attempts</h1>

In [5]:
import bcrypt
import time

# creates a password which is than hashed
password = b"secure123"
hashed_password = bcrypt.hashpw(password, bcrypt.gensalt())

print("Stored hashed password:", hashed_password)


# than a function that represents a brute force attack with the user trying to attempt a login into the network
def brute_force_attack(hashed_pw, attempts):
    print("\nBrute forcer function working")
    for attempt in attempts:
        print(f"Trying password: {attempt}")

        # Hash the guessed password and compare
        if bcrypt.checkpw(attempt.encode(), hashed_pw):
            print("password is cracked", attempt)
            return attempt

        time.sleep(0.1)  # simulate attack speed 

    print("password not found in wordlist.")
    return None


wordlist = ["pass123", "admin", "letmein", "secure123", "hello123"]

brute_force_attack(hashed_password, wordlist)

class LoginSystem:
    def __init__(self, hashed_pw):
        self.hashed_pw = hashed_pw
        self.failed_attempts = 0
        self.locked_until = 0

    def login(self, attempt):
        current_time = time.time()

        # If account is locked than the you would need to try again laster
        if current_time < self.locked_until:
            print("account locked, pls try later")
            return False

        if bcrypt.checkpw(attempt.encode(), self.hashed_pw):
            print("login successful")
            self.failed_attempts = 0
            return True
        else:
            print("wrong password.")
            self.failed_attempts += 1

            # Lock account after 3 attempts
            if self.failed_attempts >= 3:
                lock_duration = 10  # lock for 10 seconds
                self.locked_until = current_time + lock_duration
                print(f"locked for {lock_duration} seconds")
                self.failed_attempts = 0

            return False

print('simulating the defence system')
system = LoginSystem(hashed_password)

# Attacker attempting multiple times
system.login("wrong1")
system.login("wrong2")
system.login("wrong3")  # triggers lockout
system.login("secure123")  # this should now be blocked due to lockout

# Wait and try again after lockout expires
time.sleep(10)
system.login("secure123")


Stored hashed password: b'$2b$12$Q7MH3kDSgta/GsArHCbDJOmKLOD6MD04ScEtJ7yxNb8ceALBrNvH2'

Brute forcer function working
Trying password: pass123
Trying password: admin
Trying password: chicken
Trying password: secure123
password is cracked secure123
simulating the defence system
wrong password.
wrong password.
wrong password.
locked for 5 seconds
account locked, pls try later
login successful


True